In [6]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm import tqdm

import joblib
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format = 'retina'

In [7]:
path = './'

In [10]:
# Fillna
def fillna_col(columns, data):
    tmp = data[columns].value_counts(dropna=True)
    max_v = max(data[columns].value_counts(dropna=True))
    max_c = tmp.loc[tmp==max_v].index[0]
    data[columns] = data[columns].fillna(max_c)
    return data

# 숫자형 변수 -> 선형보간하는 함수
def fillna_val(columns, data):
    data[columns] = data[columns].interpolate(method='linear')
    return data

def fillna_total(data):
    tmp_n = data.isna().sum()
    nan_list = list(tmp_n[tmp_n>0].index)
    for columns in nan_list:
        if data[columns].dtype == 'O':
            fillna_col(columns, data)
        else:
            fillna_val(columns, data)
    return data

# 필요한 데이터 추가
def add_var(data):
    # 준공연도
    data['준공일자'] = data['준공일자'].astype('str')
    data['준공연도'] = data['준공일자'].str[:4]
    data['준공연도'] = data['준공연도'].astype(np.dtype("int64"))

    # 총 면적
    data['총면적'] = (data['전용면적'] + data['공용면적']) * data['전용면적별세대수'] 
    return data

# 필요없는 데이터 삭제
def remove_var(data, col):
    data.drop(col, axis=1, inplace=True)
    return data
    
# 단지별 데이터 분리 및 처리
def data01_process(data, mode):
    if mode == 'test' or mode == 'test_dummy':
        data01 = data[['단지코드', '총세대수', '지역', '준공연도', '건물형태', '난방방식', '승강기설치여부']]
    else:
        data01 = data[['단지코드', '총세대수', '지역', '준공연도', '건물형태', '난방방식', '승강기설치여부', '실차량수']]
    len_1 = len(data01)
    data01 = data01.drop_duplicates()
    data01.reset_index()
    len_2 = len(data01)
    print(f'number of before data01 processing: {len_1}')
    print(f'number of after data01 processing: {len_2}')
    return data01

# 상세데이터 집계 1: 단지코드별 총면적 합 집계
def data02_process1(data, col1, col2):
    data02 = data[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]
    df_area = data02.groupby(col1, as_index=False)[col2].sum()
    return df_area

# 상세데이터 집계 2: 전용면적 구간별 집계 (피벗 형태)
## 1: Applicate jenks natural breaks 
def data02_process2_jenks(data, col1, col2, n):
    data02 = data[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]
    bins = [0]
    bins = bins + jenkspy.jenks_breaks(data['전용면적'], n_classes=n)
    labels = [f'면적{bins[i]}_{bins[i+1]}' for i in range(0, len(bins)-1)]
    data02['전용면적구간'] = pd.cut(data02[col2], bins = bins, labels = labels, right=False)
    tmp = data02.groupby([col1, '전용면적구간'], as_index = False)['전용면적구간'].value_counts()
    df_pivot = tmp.pivot(index=col1, columns='전용면적구간')
    df_pivot[labels] = df_pivot['count'][labels]
    df_pivot = df_pivot.drop('count', axis=1)
    df_pivot[col1] = list(df_pivot.index)
    df_pivot.reset_index(drop=True, inplace=True)
    temp_ = pd.DataFrame()
    temp_['단지코드'] = df_pivot['단지코드']
    temp_[labels] = df_pivot[labels]
    df_pivot = temp_
    return df_pivot
## 2: Applicate Users bin 

def data02_process2(data, col1, col2, bin):
    data02 = data[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]
    bins = bin
    labels = [f'면적{bins[i]}_{bins[i+1]}' for i in range(0, len(bins)-1)]
    data02['전용면적구간'] = pd.cut(data02[col2], bins = bins, labels = labels, right=False)
    tmp = data02.groupby([col1, '전용면적구간'], as_index = False)['전용면적구간'].value_counts()
    df_pivot = tmp.pivot(index=col1, columns='전용면적구간')
    df_pivot[labels] = df_pivot['count'][labels]
    df_pivot = df_pivot.drop('count', axis=1)
    df_pivot[col1] = list(df_pivot.index)
    df_pivot.reset_index(drop=True, inplace=True)
    temp_ = pd.DataFrame()
    temp_['단지코드'] = df_pivot['단지코드']
    temp_[labels] = df_pivot[labels]
    df_pivot = temp_
    return df_pivot

# 상세데이터 집계 2: 임대료 및 임대보증금 평균
def data02_process3(data):
    data02 = data[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]
    df_rent = data02.groupby('단지코드', as_index = False)['임대보증금'].mean()
    df_rent['임대료'] = data02.groupby('단지코드', as_index = False)['임대료'].mean()['임대료']
    return df_rent

In [12]:
# 입력 데이터 전처리 파이프 라인 proto type
def preprocessing_input_data(data, bin, mode):
    # 결측치 보간
    data = fillna_total(data)
    data = add_var(data)
    remove_col = ['단지명', '단지내주차면수', '준공일자']
    data = remove_var(data, remove_col)
    data01 = data01_process(data, mode)
    df_area = data02_process1(data, '단지코드', '총면적')
    #df_pivot = data02_process2_jenks(data, '단지코드', '전용면적', 5)
    df_pivot = data02_process2(data, '단지코드', '전용면적', bin)
    df_rent = data02_process3(data)
    temp1 = pd.merge(df_area, df_rent, on='단지코드', how='left')
    temp2 = pd.merge(temp1, df_pivot, how='left', on='단지코드')
    base_data = pd.merge(data01, temp2, how='left', on='단지코드')
    base_data['난방방식'] = base_data['난방방식'].map({'개별가스난방': '개별', '개별유류난방': '개별',
                                     '지역난방': '지역', '지역가스난방': '지역',
                                     '지역유류난방': '지역', '중앙가스난방': '중앙',
                                     '중앙난방': '중앙', '중앙유류난방': '중앙'})
    base_data['승강기설치여부'] = base_data['승강기설치여부'].map({'전체동 설치': 1, '일부동 설치': 0, '미설치': 0})
    base_data.drop(columns=['단지코드', '지역'], inplace=True)
    joblib.dump(base_data, path+f'base_data_{mode}.pkl')
    print('Input Data Preprocessing is Complete!!')

In [ ]:
# 파이프라인 실행 확인: train 
bin = [10, 30, 40, 50, 60, 70, 80, 200]
for mode in ['train','test_dummy']:
    apart = pd.read_excel(path+f'{mode}.xlsx')
    preprocessing_input_data(apart, bin, mode)

In [683]:
def preprocessing_train_data(data, cols, dumm_cols, mode, scaler):
    if mode == 'train':
        target = '실차량수'
        x = data.drop(columns=target)
        y = data.loc[:, target]
        x = x[cols]
    # 종속변수 선택
    else:
        x = data[cols]
    
    # 가변수화
    if len(dumm_cols) > 0:
        tmp = []
        for col in dumm_cols:
            if col in cols:
                tmp.append(col)
            # 가변수화
        x = pd.get_dummies(x, columns=tmp, drop_first=True, dtype=int)
    if mode == 'train':    
        x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2)
        x_train_s = scaler.fit_transform(x_train)
        x_val_s = scaler.transform(x_val)
        return x_train, x_val, y_train, y_val, x_train_s, x_val_s, scaler
    
    else:
        x_test = x.copy()
        x_test_s = scaler.transform(x_test)
        return x_test, x_test_s

In [685]:
mode = 'train'
apart = joblib.load(path+f'base_data_{mode}.pkl')
cols = ['총세대수', '난방방식', '승강기설치여부', '총면적']
dumm_cols = ['건물형태', '난방방식']
scaler = MinMaxScaler()
x_train, x_val, y_train, y_val, x_train_s, x_val_s, scaler = preprocessing_train_data(apart, cols, dumm_cols, mode, scaler)

In [688]:
def set_result(model_name, r2, mae, r2_result, mae_result):
    r2_result[model_name] = r2
    mae_result[model_name] = mae
    return r2_result, mae_result

In [689]:
def CompareML_basic(x_train, x_val, y_train, y_val, x_train_s, x_val_s):
    r2_result = {}
    mae_result = {}

    # LinearRegression
    model_name = 'LinearRegression'
    model = LinearRegression()
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)
    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    #joblib.dump(model, f'{model_name}.pkl')
    r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
    #print(f'{model_name} complete!!')
    
    # DecisionTree
    model_name = 'DecisionTree'
    model = DecisionTreeRegressor()
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)
    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    #joblib.dump(model, f'{model_name}.pkl')
    r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
    #print(f'{model_name} complete!!')

    # KNeibor
    model_name = 'KNeibor'
    model = KNeighborsRegressor()
    model.fit(x_train_s, y_train)
    y_pred = model.predict(x_val_s)
    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    #joblib.dump(model, f'{model_name}.pkl')
    r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result) 
    #print(f'{model_name} complete!!')

    # RandomForest
    model_name = 'RandomForest'
    model = RandomForestRegressor()
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)
    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    #joblib.dump(model, f'{model_name}.pkl')
    r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
    #print(f'{model_name} complete!!')

    # XGBoost
    model_name = 'XGBoost'
    model = XGBRegressor()
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)
    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    #joblib.dump(model, f'{model_name}.pkl')
    r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
    #print(f'{model_name} complete!!')

    # LGBM
    '''model_name = 'LGBM'
    model = LGBMRegressor()
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)
    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    #joblib.dump(model, f'{model_name}.pkl')
    r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
    print(f'{model_name} complete!!')'''

    result_df = pd.DataFrame()
    result_df['r2'] = r2_result
    result_df['MAE'] = mae_result
    
    return result_df

In [696]:
# iter 만큼 무작위 모델을 생성한 후, 모델 별 평균 r2 score 비교
result_list = []
model_list = ['LinearRegression', 'DecisionTree', 'KNeibor', 'RandomForest', 'XGBoost']#, 'LGBM']
r2_dict = {}
mae_dict = {}
iter = 500
for i in tqdm(range(1, iter+1)):
    result_df = CompareML_basic(x_train, x_val, y_train, y_val, x_train_s, x_val_s)
    result_list.append(result_df)

for model_name in model_list:
    tmp = []
    for df in result_list:
        tmp.append(df['r2'].loc[model_name])
    score = sum(tmp)/len(tmp)
    r2_dict[model_name] = score

df_r2_mean = pd.DataFrame(r2_dict.values(), columns=['r2 Mean'], index=model_list)
df_r2_mean.sort_values(by = 'r2 Mean', ascending=False, inplace=True)
best_model = df_r2_mean.index[0]
print(f'The best relable model is {best_model}!!')
display(df_r2_mean)

100%|███████████████████████████████████████████████████████████████████████████████████████| 500/500 [02:51<00:00,  2.92it/s]


In [724]:
def ML_tune(x_train, x_val, y_train, y_val, x_train_s, x_val_s, model_name):
    r2_result = {}
    mae_result = {}

    # LinearRegression
    if model_name == 'LinearRegression':
        model = LinearRegression()
        model.fit(x_train, y_train)
        y_pred = model.predict(x_val)
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
        print(f'{model_name} complete!!')
    
    # DecisionTree
    if model_name == 'DecisionTree':
        param = {'max_depth': [10, 20, 30, 40, 50],
                'min_samples_split': [2, 5, 10, 20, 50],
                'min_samples_leaf': [1, 2, 5, 10, 20]}
        model_ = DecisionTreeRegressor(max_depth=5)
        model = GridSearchCV(model_, param, cv=5, scoring='r2')
        model.fit(x_train, y_train)
        y_pred = model.predict(x_val)
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
        print(f'{model_name} complete!!')

    # KNeibor
    if model_name == 'KNeibor':
        param = {'n_neighbors': range(1, 10)}
        model_ = KNeighborsRegressor()
        model = GridSearchCV(model_, param, cv=5, scoring='r2')
        model.fit(x_train_s, y_train)
        y_pred = model.predict(x_val_s)
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result) 
        print(f'{model_name} complete!!')

    # RandomForest
    if model_name == 'RandomForest':
        param = {'max_depth': [10, 20, 30, 40, 50],
                'min_samples_split':[2,5,10,20,50],
                'min_samples_leaf': [1, 2, 5, 10, 20],
                'n_estimators': [100, 200, 300]}
        model_ = RandomForestRegressor()
        model = GridSearchCV(model_, param, cv=5, scoring='r2')
        model.fit(x_train, y_train)
        y_pred = model.predict(x_val)
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
        print(f'{model_name} complete!!')

    # XGBoost
    if model_name == 'XGBoost':
        model_ = XGBRegressor()
        param = {'n_estimators': [100, 200, 300],
                      'learning_rate': [0.01, 0.05, 0.1, 0.2],
                      'max_depth': [3, 5, 7, 10],
                      'subsample': [0.6, 0.8, 1.0],
                      'colsample_bytree': [0.6, 0.8, 1.0],
                      'gamma': [0, 1, 5],
                      'reg_alpha': [0, 0.1, 0.5],
                      'reg_lambda': [1, 1.5, 2]}
        model = GridSearchCV(model_, param_grid=param, cv=5, scoring='r2')
        model.fit(x_train, y_train)
        y_pred = model.predict(x_val)
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
        print(f'{model_name} complete!!')

    # LGBM
    '''if model_name == 'LGBM':
        model_ = LGBMRegressor()
        model = GridSearchCV(model_, cv=5, scoring='r2')
        model.fit(x_train, y_train)
        y_pred = model.predict(x_val)
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        r2_result, mae_result = set_result(model_name, r2, mae, r2_result, mae_result)
        print(f'{model_name} complete!!')'''

    result_df = pd.DataFrame()
    result_df[f'r2'] = r2_result
    result_df[f'MAE'] = mae_result
    joblib.dump(model, f'{model_name}.pkl')
    
    return result_df

In [726]:
result_df = ML_tune(x_train, x_val, y_train, y_val, x_train_s, x_val_s, best_model)

XGBoost complete!!


In [718]:
def predict_target(data, best_model, x_test, x_test_s):
    model = joblib.load(f'{best_model}.pkl')
        
    # KNeibor
    if model_name == 'KNeibor':
        y_pred = model.predict(x_test_s)
    else:
        y_pred = model.predict(x_test)

    data['실차량수 예측값'] = y_pred
    return data

In [719]:
mode = 'test_dummy'
apart = joblib.load(path+f'base_data_{mode}.pkl')
#x_test, x_test_s = preprocessing_models_data(apart, cols, dumm_cols, mode, scaler)
#data = predict_target(apart, best_model, x_test, x_test_s)